In [1]:
import cv2
import tkinter as tk
from tkinter import filedialog, messagebox
from ultralytics import YOLO
from PIL import Image, ImageTk

class ObjectDetectionApp:
    def __init__(self, window):
        self.window = window
        self.window.title("YOLO Object Detection")
        self.window.geometry("800x600")
        self.window.config(bg="#2C3E50")  # Set background color

        # Load YOLOv8 model
        self.model = YOLO('yolov8n.pt')  # You can use 'yolov8s.pt' or 'yolov8m.pt' for higher accuracy

        # Initialize video capture
        self.cap = None
        self.is_video_mode = False

        # Create a frame for the webcam feed
        self.video_frame = tk.Frame(self.window, bg="#34495E", bd=5)
        self.video_frame.pack(pady=20)

        # Create a Label to display frames
        self.frame_label = tk.Label(self.video_frame, bg="#34495E")
        self.frame_label.pack()

        # Create a footer frame for buttons
        self.footer_frame = tk.Frame(self.window, bg="#2C3E50")
        self.footer_frame.pack(side="bottom", pady=20)

        # Create a dropdown menu for selecting mode
        self.mode_label = tk.Label(self.footer_frame, text="Select Mode:", font=("Helvetica", 12), fg="white", bg="#2C3E50")
        self.mode_label.pack(side="left", padx=10)

        self.mode_var = tk.StringVar(value="Video Feed")
        self.mode_menu = tk.OptionMenu(self.footer_frame, self.mode_var, "Video Feed", "Image", "Video File")
        self.mode_menu.pack(side="left", padx=10)

        # Create a start detection button with style
        self.start_button = tk.Button(self.footer_frame, text="Start Detection", command=self.start_detection, 
                                      font=("Helvetica", 14), fg="white", bg="#1ABC9C", relief="raised", bd=5, width=20)
        self.start_button.pack(pady=10)

        # Create a quit button with style
        self.quit_button = tk.Button(self.footer_frame, text="Quit", command=self.quit_application,
                                      font=("Helvetica", 14), fg="white", bg="#E74C3C", relief="raised", bd=5, width=20)
        self.quit_button.pack(pady=10)

        # Add a label for instructions
        self.instruction_label = tk.Label(self.window, text="Press 'Start Detection' to begin", 
                                          font=("Helvetica", 12), fg="#ECF0F1", bg="#2C3E50")
        self.instruction_label.pack()

        # Add a splash screen effect
        self.show_splash_screen()

    def show_splash_screen(self):
        splash_label = tk.Label(self.window, text="YOLO Object Detection", font=("Helvetica", 24, "bold"), 
                                fg="#ECF0F1", bg="#2C3E50")
        splash_label.place(relx=0.5, rely=0.4, anchor="center")
        self.window.after(2000, splash_label.destroy)  # Hide splash screen after 2 seconds

    def start_detection(self):
        mode = self.mode_var.get()
        
        if mode == "Video Feed":
            self.start_video_feed()
        elif mode == "Image":
            self.load_image()
        elif mode == "Video File":
            self.load_video_file()

    def start_video_feed(self):
        # Initialize webcam for live video feed
        self.cap = cv2.VideoCapture(0)
        if not self.cap.isOpened():
            messagebox.showerror("Error", "Could not access the webcam.")
            return

        print("Starting detection from Video Feed...")
        self.detect_objects()

    def load_image(self):
        # Open file dialog to select an image
        file_path = filedialog.askopenfilename(title="Select an Image", filetypes=[("Image Files", "*.png;*.jpg;*.jpeg")])
        if file_path:
            image = cv2.imread(file_path)
            if image is not None:
                print("Starting detection from Image...")
                self.detect_objects_from_image(image)

    def load_video_file(self):
        # Open file dialog to select a video file
        file_path = filedialog.askopenfilename(title="Select a Video", filetypes=[("Video Files", "*.mp4;*.avi")])
        if file_path:
            self.cap = cv2.VideoCapture(file_path)
            if not self.cap.isOpened():
                messagebox.showerror("Error", "Could not open the video file.")
                return
            print("Starting detection from Video File...")
            self.detect_objects_from_video()

    def detect_objects(self):
        # Detect objects from webcam feed (Video Feed mode)
        while True:
            ret, frame = self.cap.read()
            if not ret:
                print("Error: Failed to read from the webcam.")
                break

            # Run YOLOv8 inference on the frame
            results = self.model(frame)

            # Annotate frame with bounding boxes, labels, etc.
            annotated_frame = results[0].plot()

            # Convert OpenCV frame (BGR) to PIL Image (RGB)
            annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(annotated_frame_rgb)

            # Convert PIL Image to ImageTk format
            img_tk = ImageTk.PhotoImage(image=pil_image)

            # Update the image displayed in the label
            self.frame_label.config(image=img_tk)
            self.frame_label.image = img_tk

            # Update the GUI
            self.window.update()

            # Add exit condition (pressing 'q' will break the loop)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    def detect_objects_from_image(self, image):
        # Detect objects from a loaded image (Image mode)
        results = self.model(image)

        # Annotate the image with bounding boxes
        annotated_image = results[0].plot()

        # Convert OpenCV frame (BGR) to PIL Image (RGB)
        annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(annotated_image_rgb)

        # Convert PIL Image to ImageTk format
        img_tk = ImageTk.PhotoImage(image=pil_image)

        # Display the annotated image
        self.frame_label.config(image=img_tk)
        self.frame_label.image = img_tk

    def detect_objects_from_video(self):
        # Detect objects from a loaded video file (Video File mode)
        while True:
            ret, frame = self.cap.read()
            if not ret:
                print("Error: Failed to read from the video.")
                break

            # Run YOLOv8 inference on the frame
            results = self.model(frame)

            # Annotate the frame with bounding boxes, labels, etc.
            annotated_frame = results[0].plot()

            # Convert OpenCV frame (BGR) to PIL Image (RGB)
            annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(annotated_frame_rgb)

            # Convert PIL Image to ImageTk format
            img_tk = ImageTk.PhotoImage(image=pil_image)

            # Update the image displayed in the label
            self.frame_label.config(image=img_tk)
            self.frame_label.image = img_tk

            # Update the GUI
            self.window.update()

            # Add exit condition (pressing 'q' will break the loop)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    def quit_application(self):
    # Release the video capture and destroy all windows
            if self.cap:
                self.cap.release()  # Releases the video feed or video file if it's open
            cv2.destroyAllWindows()  # Closes any OpenCV windows that may still be open
            self.window.quit()  # Closes the Tkinter window and terminates the application


if __name__ == "__main__":
    root = tk.Tk()
    app = ObjectDetectionApp(root)
    root.mainloop()


Starting detection from Video Feed...

0: 480x640 1 person, 1 clock, 85.6ms
Speed: 5.0ms preprocess, 85.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 clock, 72.7ms
Speed: 0.0ms preprocess, 72.7ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 clock, 66.2ms
Speed: 1.4ms preprocess, 66.2ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 clock, 63.2ms
Speed: 2.0ms preprocess, 63.2ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 clock, 75.5ms
Speed: 0.0ms preprocess, 75.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 clock, 72.5ms
Speed: 0.8ms preprocess, 72.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 clock, 74.8ms
Speed: 1.9ms preprocess, 74.8ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 per

Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\Rutuja\AppData\Local\Programs\Python\Python311\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Rutuja\AppData\Local\Temp\ipykernel_28100\1637392345.py", line 69, in start_detection
    self.start_video_feed()
  File "C:\Users\Rutuja\AppData\Local\Temp\ipykernel_28100\1637392345.py", line 83, in start_video_feed
    self.detect_objects()
  File "C:\Users\Rutuja\AppData\Local\Temp\ipykernel_28100\1637392345.py", line 124, in detect_objects
    img_tk = ImageTk.PhotoImage(image=pil_image)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rutuja\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\ImageTk.py", line 128, in __init__
    self.__photo = tkinter.PhotoImage(**kw)
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rutuja\AppData\Local\Programs\Python\Python311\Lib\tkinter\__init__.py", line 

In [2]:
import cv2
import tkinter as tk
from tkinter import filedialog, messagebox
from ultralytics import YOLO
from PIL import Image, ImageTk

class ObjectDetectionApp:
    def __init__(self, window):
        self.window = window
        self.window.title("YOLO Object Detection")
        self.window.geometry("800x600")
        self.window.config(bg="#2C3E50")  # Set background color

        # Load YOLOv8 model
        self.model = YOLO('yolov8n.pt')  # You can use 'yolov8s.pt' or 'yolov8m.pt' for higher accuracy

        # Initialize video capture
        self.cap = None
        self.is_video_mode = False

        # Create a frame for the webcam feed
        self.video_frame = tk.Frame(self.window, bg="#34495E", bd=5)
        self.video_frame.pack(pady=20)

        # Create a Label to display frames
        self.frame_label = tk.Label(self.video_frame, bg="#34495E")
        self.frame_label.pack()

        # Create a footer frame for buttons
        self.footer_frame = tk.Frame(self.window, bg="#2C3E50")
        self.footer_frame.pack(side="bottom", pady=20)

        # Create a dropdown menu for selecting mode
        self.mode_label = tk.Label(self.footer_frame, text="Select Mode:", font=("Helvetica", 12), fg="white", bg="#2C3E50")
        self.mode_label.pack(side="left", padx=10)

        self.mode_var = tk.StringVar(value="Video Feed")
        self.mode_menu = tk.OptionMenu(self.footer_frame, self.mode_var, "Video Feed", "Image", "Video File")
        self.mode_menu.pack(side="left", padx=10)

        # Create a start detection button with style
        self.start_button = tk.Button(self.footer_frame, text="Start Detection", command=self.start_detection, 
                                      font=("Helvetica", 14), fg="white", bg="#1ABC9C", relief="raised", bd=5, width=20)
        self.start_button.pack(pady=10)

        # Create a quit button with style
        self.quit_button = tk.Button(self.footer_frame, text="Quit", command=self.quit_application,
                                      font=("Helvetica", 14), fg="white", bg="#E74C3C", relief="raised", bd=5, width=20)
        self.quit_button.pack(pady=10)

        # Add a label for instructions
        self.instruction_label = tk.Label(self.window, text="Press 'Start Detection' to begin", 
                                          font=("Helvetica", 12), fg="#ECF0F1", bg="#2C3E50")
        self.instruction_label.pack()

        # Add a splash screen effect
        self.show_splash_screen()

    def show_splash_screen(self):
        splash_label = tk.Label(self.window, text="YOLO Object Detection", font=("Helvetica", 24, "bold"), 
                                fg="#ECF0F1", bg="#2C3E50")
        splash_label.place(relx=0.5, rely=0.4, anchor="center")
        self.window.after(2000, splash_label.destroy)  # Hide splash screen after 2 seconds

    def start_detection(self):
        mode = self.mode_var.get()
        
        if mode == "Video Feed":
            self.start_video_feed()
        elif mode == "Image":
            self.load_image()
        elif mode == "Video File":
            self.load_video_file()

    def start_video_feed(self):
        # Initialize webcam for live video feed
        self.cap = cv2.VideoCapture(0)
        if not self.cap.isOpened():
            messagebox.showerror("Error", "Could not access the webcam.")
            return

        print("Starting detection from Video Feed...")
        self.detect_objects()

    def load_image(self):
        # Open file dialog to select an image
        file_path = filedialog.askopenfilename(title="Select an Image", filetypes=[("Image Files", "*.png;*.jpg;*.jpeg")])
        if file_path:
            image = cv2.imread(file_path)
            if image is not None:
                print("Starting detection from Image...")
                self.detect_objects_from_image(image)

    def load_video_file(self):
        # Open file dialog to select a video file
        file_path = filedialog.askopenfilename(title="Select a Video", filetypes=[("Video Files", "*.mp4;*.avi")])
        if file_path:
            self.cap = cv2.VideoCapture(file_path)
            if not self.cap.isOpened():
                messagebox.showerror("Error", "Could not open the video file.")
                return
            print("Starting detection from Video File...")
            self.detect_objects_from_video()

    def detect_objects(self):
        # Detect objects from webcam feed (Video Feed mode)
        while True:
            ret, frame = self.cap.read()
            if not ret:
                print("Error: Failed to read from the webcam.")
                break

            # Run YOLOv8 inference on the frame
            results = self.model(frame)

            # Annotate frame with bounding boxes, labels, but exclude the confidence score
            annotated_frame = results[0].plot(labels=True, conf=False)

            # Convert OpenCV frame (BGR) to PIL Image (RGB)
            annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(annotated_frame_rgb)

            # Convert PIL Image to ImageTk format
            img_tk = ImageTk.PhotoImage(image=pil_image)

            # Update the image displayed in the label
            self.frame_label.config(image=img_tk)
            self.frame_label.image = img_tk

            # Update the GUI
            self.window.update()

            # Add exit condition (pressing 'q' will break the loop)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    def detect_objects_from_image(self, image):
        # Detect objects from a loaded image (Image mode)
        results = self.model(image)

        # Annotate the image with bounding boxes, labels, but exclude the confidence score
        annotated_image = results[0].plot(labels=True, conf=False)

        # Convert OpenCV frame (BGR) to PIL Image (RGB)
        annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(annotated_image_rgb)

        # Convert PIL Image to ImageTk format
        img_tk = ImageTk.PhotoImage(image=pil_image)

        # Display the annotated image
        self.frame_label.config(image=img_tk)
        self.frame_label.image = img_tk

    def detect_objects_from_video(self):
        # Detect objects from a loaded video file (Video File mode)
        while True:
            ret, frame = self.cap.read()
            if not ret:
                print("Error: Failed to read from the video.")
                break

            # Run YOLOv8 inference on the frame
            results = self.model(frame)

            # Annotate the frame with bounding boxes, labels, but exclude the confidence score
            annotated_frame = results[0].plot(labels=True, conf=False)

            # Convert OpenCV frame (BGR) to PIL Image (RGB)
            annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(annotated_frame_rgb)

            # Convert PIL Image to ImageTk format
            img_tk = ImageTk.PhotoImage(image=pil_image)

            # Update the image displayed in the label
            self.frame_label.config(image=img_tk)
            self.frame_label.image = img_tk

            # Update the GUI
            self.window.update()

            # Add exit condition (pressing 'q' will break the loop)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    def quit_application(self):
        # Release the video capture and destroy all windows
        if self.cap:
            self.cap.release()  # Releases the video feed or video file if it's open
        cv2.destroyAllWindows()  # Closes any OpenCV windows that may still be open
        self.window.quit()  # Closes the Tkinter window and terminates the application


if __name__ == "__main__":
    root = tk.Tk()
    app = ObjectDetectionApp(root)
    root.mainloop()


Starting detection from Image...

0: 384x640 13 persons, 96.3ms
Speed: 2.6ms preprocess, 96.3ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Starting detection from Image...

0: 448x640 4 persons, 1 backpack, 3 chairs, 1 tv, 1 laptop, 1 remote, 89.2ms
Speed: 2.8ms preprocess, 89.2ms inference, 0.0ms postprocess per image at shape (1, 3, 448, 640)
Starting detection from Video Feed...

0: 480x640 4 persons, 108.0ms
Speed: 0.0ms preprocess, 108.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 91.0ms
Speed: 6.3ms preprocess, 91.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 89.4ms
Speed: 0.0ms preprocess, 89.4ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 78.8ms
Speed: 0.0ms preprocess, 78.8ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 78.3ms
Speed: 0.0ms preprocess, 78.3ms inference, 0.0ms post

Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\Rutuja\AppData\Local\Programs\Python\Python311\Lib\tkinter\__init__.py", line 1948, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\Rutuja\AppData\Local\Temp\ipykernel_27036\692495371.py", line 69, in start_detection
    self.start_video_feed()
  File "C:\Users\Rutuja\AppData\Local\Temp\ipykernel_27036\692495371.py", line 83, in start_video_feed
    self.detect_objects()
  File "C:\Users\Rutuja\AppData\Local\Temp\ipykernel_27036\692495371.py", line 124, in detect_objects
    img_tk = ImageTk.PhotoImage(image=pil_image)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rutuja\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\ImageTk.py", line 128, in __init__
    self.__photo = tkinter.PhotoImage(**kw)
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rutuja\AppData\Local\Programs\Python\Python311\Lib\tkinter\__init__.py", line 413